In [ ]:
"""
generate_traffic_clustered.py
Generates synthetic datacenter network traffic traces for HTSim.
"""

from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np

WORKLOAD_FILES = {
    "DM": "../_flow_dis/DM.csv",
    "HD": "../_flow_dis/HD.csv",
    "WS": "../_flow_dis/WS.csv",
}

BYTES_PER_GBIT = 10e8 / 8  # bytes per second per Gbps


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

@dataclass
class TrafficConfig:
    n_racks: int
    hosts_per_rack: int
    load_fraction: float       # fraction of link capacity (0–1)
    duration_s: float          # simulation duration in seconds
    link_rate_gbps: float      # link rate in Gbps
    active_host_fraction: float  # fraction of hosts that generate traffic (0–1)
    workload: str              # "DM", "HD", or "WS"
    network: str               # output subdirectory name (e.g. "opera")
    seed: int
    n_groups: int = 1          # number of rack groups (1 = uniform/no grouping)

    @property
    def n_hosts(self) -> int:
        return self.n_racks * self.hosts_per_rack

    @property
    def n_active_hosts(self) -> int:
        return int(np.ceil(self.n_hosts * self.active_host_fraction))

    @property
    def link_rate_bps(self) -> float:
        return self.link_rate_gbps * BYTES_PER_GBIT

    def output_path(self) -> Path:
        fname = (
            f"{self.workload}"
            f"_{self.n_groups}Ngroup"
            f"_{100 * self.load_fraction:.2f}percLoad"
            f"_{int(self.duration_s)}sec"
            f"_{self.n_racks}N_{self.hosts_per_rack}hpr"
            f"_{self.n_hosts}hosts"
            f"_{self.link_rate_gbps}Gbps"
            f"_{self.active_host_fraction:.2f}Nactive"
            f"_seed={self.seed}.htsim"
        )
        return Path(self.network) / fname


# ---------------------------------------------------------------------------
# Pipeline steps
# ---------------------------------------------------------------------------

def set_seeds(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)


def generate_srcdst_pairs_uniform(cfg: TrafficConfig) -> tuple[np.ndarray, np.ndarray]:
    """
    Return all valid inter-rack (src, dst) pairs and their uniform CDF.
    Flows are only generated between hosts on *different* racks.
    """
    n = cfg.n_active_hosts
    rack_of = lambda h: h // cfg.hosts_per_rack  # noqa: E731

    pairs = [
        (src, dst)
        for src in range(n)
        for dst in range(n)
        if rack_of(src) != rack_of(dst)
    ]

    srcdst = np.array(pairs, dtype=int)
    cdf = np.linspace(1 / len(pairs), 1.0, len(pairs))
    return srcdst, cdf


def generate_srcdst_pairs_grouped(
    cfg: TrafficConfig,
    n_groups: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Return inter-rack src/dst pairs restricted to hosts within the same group.

    The n_racks racks are divided into n_groups equal-sized groups.
    A flow is valid if:
      - src and dst are in the same group, AND
      - src and dst are on different racks (different ToRs)

    Parameters
    ----------
    cfg      : TrafficConfig
    n_groups : number of groups to partition racks into (must divide n_racks evenly)

    Returns
    -------
    srcdst   : (N, 2) int array of valid [src, dst] host indices
    cdf      : (N,)  float array — uniform CDF over all valid pairs
    """
    if cfg.n_racks % n_groups != 0:
        raise ValueError(
            f"n_groups={n_groups} must evenly divide n_racks={cfg.n_racks}"
        )

    racks_per_group = cfg.n_racks // n_groups
    n = cfg.n_active_hosts

    rack_of  = lambda h: h // cfg.hosts_per_rack        # noqa: E731
    group_of = lambda h: rack_of(h) // racks_per_group  # noqa: E731

    pairs = [
        (src, dst)
        for src in range(n)
        for dst in range(n)
        if group_of(src) == group_of(dst)  # same group
        and rack_of(src) != rack_of(dst)   # different ToR
    ]

    if not pairs:
        raise ValueError(
            "No valid pairs found — check n_groups vs hosts_per_rack."
        )

    srcdst = np.array(pairs, dtype=int)
    cdf    = np.linspace(1 / len(pairs), 1.0, len(pairs))
    return srcdst, cdf


def load_flow_size_distribution(workload: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Load a flow-size CDF from a two-column CSV (size_bytes, cdf_value).
    Returns (flowsize, flowcdf) arrays.
    """
    if workload not in WORKLOAD_FILES:
        raise ValueError(f"Unknown workload '{workload}'. Choose from {list(WORKLOAD_FILES)}")

    data = np.loadtxt(WORKLOAD_FILES[workload], delimiter=",")
    return data[:, 0], data[:, 1]


def compute_arrival_rate(
    cfg: TrafficConfig,
    flowsize: np.ndarray,
    flowcdf: np.ndarray,
) -> float:
    """
    Compute the network-wide Poisson flow arrival rate (flows/second)
    that achieves the requested load fraction.
    """
    avg_flowsize = np.sum(flowsize[1:] * np.diff(flowcdf))  # bytes/flow
    lambda_per_host = (cfg.load_fraction * cfg.link_rate_bps) / avg_flowsize
    return cfg.n_active_hosts * lambda_per_host


def generate_flow_arrivals(duration_s: float, arrival_rate: float) -> np.ndarray:
    """
    Draw Poisson-process arrival times via exponential inter-arrivals.
    Returns a 1-D array of arrival times in nanoseconds (int64).
    """
    # Pre-allocate generously; trim at the end
    n_est = int(np.ceil(arrival_rate * duration_s * 1.2))
    times_ns = []
    t = 0.0
    while t < duration_s:
        t += -np.log(1 - np.random.rand()) / arrival_rate
        times_ns.append(int(t * 1e9))

    return np.array(times_ns, dtype=np.int64)


def assign_flow_attributes(
    arrival_times_ns: np.ndarray,
    flowsize: np.ndarray,
    flowcdf: np.ndarray,
    srcdst: np.ndarray,
    srcdst_cdf: np.ndarray,
) -> np.ndarray:
    """
    Assign a source, destination, and size to every flow via inverse-CDF sampling.

    Returns an (N, 4) int64 array: [src, dst, size_bytes, start_ns].
    """
    n = len(arrival_times_ns)
    flowmat = np.zeros((n, 4), dtype=np.int64)
    flowmat[:, 3] = arrival_times_ns

    # Sample flow sizes from CDF
    rand_sizes = np.random.rand(n)
    size_idx = np.searchsorted(flowcdf, rand_sizes)
    flowmat[:, 2] = flowsize[size_idx]

    # Sample src/dst pairs from uniform CDF
    rand_pairs = np.random.rand(n)
    pair_idx = np.searchsorted(srcdst_cdf, rand_pairs)
    flowmat[:, 0:2] = srcdst[pair_idx]

    return flowmat


def write_htsim_file(flowmat: np.ndarray, path: Path) -> None:
    """Write flows to an HTSim trace file (space-separated: src dst size start_ns)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for i, (src, dst, size, start_ns) in enumerate(flowmat):
            line = f"{src} {dst} {size} {start_ns}"
            f.write(line if i == len(flowmat) - 1 else line + "\n")
    print(f"Written {len(flowmat):,} flows → {path}")


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def generate_traffic(cfg: TrafficConfig) -> None:
    """Full pipeline: config → topology → arrivals → attributes → file."""
    set_seeds(cfg.seed)

    print(f"Config: {cfg}")
    print(f"Active hosts: {cfg.n_active_hosts} / {cfg.n_hosts}")

    srcdst, srcdst_cdf = generate_srcdst_pairs_grouped(cfg, cfg.n_groups)
    flowsize, flowcdf = load_flow_size_distribution(cfg.workload)
    arrival_rate = compute_arrival_rate(cfg, flowsize, flowcdf)

    arrival_times_ns = generate_flow_arrivals(cfg.duration_s, arrival_rate)
    flowmat = assign_flow_attributes(arrival_times_ns, flowsize, flowcdf, srcdst, srcdst_cdf)

    actual_load = np.sum(flowmat[:, 2]) / (cfg.duration_s * cfg.n_hosts * cfg.link_rate_bps)
    print(f"Requested load: {cfg.load_fraction:.3f}  |  Actual load: {actual_load:.3f}")

    write_htsim_file(flowmat, cfg.output_path())


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    BASE = dict(
        n_racks=108,
        hosts_per_rack=6,
        duration_s=10.001,
        link_rate_gbps=40,
        active_host_fraction=1.0,
        n_groups=12,
        workload="HD",
        network="opera",
        seed=1,
    )

    for load in [0.02, 0.05, 0.10, 0.20]:
        cfg = TrafficConfig(load_fraction=load, **BASE)
        generate_traffic(cfg)